In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

### Understanding the final wrap up of a transformer.

`self.out_proj(x)` In this part of the transformer.

Let's think of tradiaitonal transformer

Input:  "The  cat  sat"


pos0 "The" → [0.1, 2.3, -1.4, 0.8, ...]   vocab_size scores

pos1 "cat" → [1.2, 0.3,  2.1, -0.5, ...]   vocab_size scores  

pos2 "sat" → [-0.2, 1.8, 0.4, 3.1, ...]   vocab_size scores

In [ ]:
# Let's run one forward pass here with the loaded skeleton to
# make sure we don't get confused.

''' Just simple example may not be the model that we are trying to analyse. '''

import sys
import time
import torch
import torch.nn as nn
from pathlib import Path
from torch.utils.data import Dataset, DataLoader, random_split

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

''''
Experimental Model 

 '''
class FibonacciModDataset(Dataset):
    def __init__(self, seq_len=10, mod=10, num_samples=10000):
        self.mod = mod
        self.global_seq = self.generate_fib_sequence(1000, mod)
        self.samples = []
        for _ in range(num_samples):
            start_idx = torch.randint(0, len(self.global_seq) - seq_len - 1, (1,)).item()
            seq = self.global_seq[start_idx:start_idx + seq_len + 1]
            x = torch.tensor(seq[:-1], dtype=torch.long)
            y = torch.tensor(seq[1:], dtype=torch.long)
            self.samples.append((x, y))

    def generate_fib_sequence(self, length, mod):
        seq = [1, 1]
        while len(seq) < length:
            seq.append((seq[-1] + seq[-2]) % mod)
        return seq

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


class MLP(nn.Module):
    def __init__(self, d_model, expansion=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_model * expansion),
            nn.GELU(),
            nn.Linear(d_model * expansion, d_model),
        )

    def forward(self, x):
        return self.net(x)


class MinimalTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=4, n_heads=1, num_layers=3, max_seq_len=20):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(max_seq_len, d_model)
        self.layers = nn.ModuleList([
            nn.MultiheadAttention(d_model, n_heads, batch_first=True)
            for _ in range(num_layers)
        ])
        self.mlps = nn.ModuleList([
            MLP(d_model)
            for _ in range(num_layers)
        ])
        self.out_proj = nn.Linear(d_model, vocab_size)

    def forward(self, tokens):
        B, T = tokens.shape
        pos = torch.arange(T, device=tokens.device)
        x = self.token_embed(tokens) + self.pos_embed(pos).unsqueeze(0)
        attn_mask = torch.triu(torch.ones(T, T, device=tokens.device) * float('-inf'), diagonal=1)
        for attn, mlp in zip(self.layers, self.mlps):
            attn_out, _ = attn(x, x, x, attn_mask=attn_mask)
            x = x + attn_out
            x = x + mlp(x)
        out_projection = self.out_proj(x)
        B, T, C = out_projection.shape  
        print(f"B {B}, SeqLen {T}, Vocab {C}")
        batch_data = out_projection[0]
        print(f"Shaoe of raw logits: {batch_data.shape}")
        print(batch_data)
        return out_projection

    def get_embeddings(self):
        return self.pos_embed + self.token_embed




def evaluate_model(model, dataloader, show_accuracy=False):
    correct, total = 0, 0
    loss_fn = nn.CrossEntropyLoss()
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            pred = logits.argmax(dim=-1)
            loss = loss_fn(logits[:, 1:].reshape(-1, logits.size(-1)), y[:, 1:].reshape(-1))
            total_loss += loss.item()
            correct += (pred[:, 1:] == y[:, 1:]).sum().item()
            total += y[:, 1:].numel()
            break # just one forward pass to analyse

    if show_accuracy:
        print(f"Accuracy (eval mode): {correct / total:.2%}")

    avg_loss = total_loss / len(dataloader)
    return avg_loss

vocab_size = 10
batch_size = 16
generated_ds = FibonacciModDataset(num_samples=25000, mod=vocab_size, seq_len=20)
train_size = int(0.8 * len(generated_ds))
test_size = len(generated_ds) - train_size
# does not matter we just need to look at output_proj
train_ds, test_ds = random_split(generated_ds, [train_size, test_size])
test_loader = DataLoader(test_ds, batch_size=batch_size, num_workers=8, pin_memory=True, persistent_workers=True, prefetch_factor=4)

model = MinimalTransformer(vocab_size=vocab_size).to(device)

evaluate_model(model, test_loader, show_accuracy=True)


B 16, SeqLen 20, Vocab 10
tensor([[-0.1279, -0.0420, -0.7120,  0.6873,  0.7029,  0.8303, -0.3060,  0.0366,
          0.3193,  0.0070],
        [-0.1398,  0.2781,  0.2180,  0.5794,  0.2376,  0.0197,  0.5074,  0.4459,
         -0.2445, -0.0669],
        [ 2.0647,  1.6007,  1.7807, -0.4721, -0.9536, -0.5647,  2.5042,  0.2330,
         -2.3766, -1.7118],
        [-2.1571, -0.1452,  1.2006,  0.5834,  0.3514, -0.2934,  1.2212,  0.8821,
          0.0930,  1.5888],
        [-0.3554, -0.7045,  1.9055,  1.4001, -1.0810, -1.1997,  4.1652,  0.7488,
         -0.2826,  1.0430],
        [-2.9916, -0.8852,  1.4930,  0.3794,  0.4771,  0.7553,  2.4691,  0.2119,
          0.4814,  2.9277],
        [-1.1402, -0.5302,  0.0300,  0.7582,  0.5583,  0.7617,  0.8759,  0.0928,
          0.5192,  1.1449],
        [-1.1398, -0.0496, -0.4344,  0.4582,  0.9932,  0.9429, -0.5162,  0.1674,
          0.4363,  0.6860],
        [-1.9917,  0.3710,  0.9577, -0.3046,  0.8145,  0.7587,  0.4915,  0.3218,
         -0.2877,  1.

0.00870511097649035

## Understanding the problem

The logits are $\ell_\theta(c \mid a, b)$

We know softmax does not change things much. And, some changes in the raw logits are not physically meaningful.

$$
\phi_c(a,b) = \ell_\theta(c \mid a, b) - \frac{1}{n}\sum_{c'=0}^{n-1} \ell_\theta(c' \mid a, b)
$$


Here with our without constant `c` subracting from average results in same.

The demo shows that two models with different arbitrary shifts in their logits produce identical φ_c(a,b). One just subracting from mean, and other adding speed of light and subracting from it's added speed of light mean.


gauge problem solved!

In [ ]:
import torch
from scipy import constants

x = torch.tensor([1.0, 2.0, 3.0], dtype=torch.float64)
logits_shifted = x + constants.c
original_centered = x - torch.mean(x)
shifted_centered = logits_shifted - torch.mean(logits_shifted)


print(shifted_centered) 
print(original_centered)
print(torch.allclose(original_centered, shifted_centered))


tensor([-1.,  0.,  1.], dtype=torch.float64)
tensor([-1.,  0.,  1.], dtype=torch.float64)
True


Centered logits are not the only way to analyze what the model learned you can also look at hidden states inside the model.


Input -> transformer -> h_θ(a,b) ∈ ℝᵈ -> logits

(a=0, b=0) → forward pass → logits [n values]

(a=0, b=1) → forward pass → logits [n values]

(a=0, b=2) → forward pass → logits [n values]


For each a, and b we have n^2 grid

Let's say we have nxn grid. a = 5, b = 3 and mod = 10


Logitis are refereing to the next token predicted by the transformer